# Combined-catalog spatial and temporal matching

Uses the Best Data columns from the final combined catalog. Four Monte Carlo tests are performed. No additional duration maximization is performed here: MJD, T90 and T90 Start are read together from the prepared catalog.

In [ ]:
# Skip this cell if the packages are already installed in your environment.
%pip install -q numpy pandas matplotlib

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re


## Configuration

Check the three input paths before selecting **Run All**. The configuration specifies the four Monte Carlo models and their trial count; runtime depends on the available hardware.

In [ ]:
# ============================================================
# Files and settings — paths are relative to the current working directory.
# ============================================================
BASE_DIR = Path(".")
PKL_FOLDER = BASE_DIR "/IceCat-2-final/dataverse_files/contours/pkl"
GRB_FILE = BASE_DIR / "Combined_GRB_Catalog.tsv"
IC_FILE = BASE_DIR / "IceCube_Gold_Bronze_Tracks.csv"

# "t90": duration-based windows; "fixed": a fixed window around MJD.
TIME_WINDOW_MODE = "t90"

# With an available T90 Start:
# [trigger + start - max(0.30*T90, 1 s),
#  trigger + start + T90 + max(0.30*T90, 1 s)]
T90_BUFFER_FRACTION = 0.30
MIN_TIME_BUFFER_SECONDS = 1.0

# Without a T90 Start:
# [trigger - max(0.40*T90, 5 s),
#  trigger + T90 + max(0.40*T90, 5 s)]
MISSING_START_BUFFER_FRACTION = 0.40
MISSING_START_MIN_SECONDS = 5.0

# Fixed window in days. This mode does not require T90 or T90 Start.
FIXED_BEFORE = 0.5
FIXED_AFTER = 0.5

# Monte Carlo settings.
N = 100000
SEED = 65789358

# Only load pickle files from a trusted source.
PKL_FILES = sorted(PKL_FOLDER.glob("IceCube-*.contours.pkl"))
ARCSEC_TO_DEG = 1.0 / 3600.0

# A fresh output directory preserves previous analysis products.
OUTPUT_DIR = BASE_DIR / ("combined_matching_" + pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f"))


## Time windows and counting

Absolute timestamps are expressed as MJD. Durations and offsets are converted from
seconds to days for time-window calculations.
With a measured T90 Start, the window is trigger + start through trigger + start + T90,
expanded at both ends by max(0.30 T90, 1 s). Without a start, use
[trigger - N, trigger + T90 + N], where N = max(0.40 T90, 5 s).
No extra 30% buffer is added to this replacement window.

In `t90` mode, a missing or non-positive duration cannot define a temporal window:
the row is listed in `excluded_rows.csv`, not silently assigned a duration.
In `fixed` mode, both duration and start may be missing. Every retained row uses the
same selected mode. No new trigger-date cut or duplicate removal is applied.

Each match is an observed source-row / IceCube-event pair satisfying BOTH conditions.
Multiple overlapping rings do not multiply the pair count. Zero matches produces an
explicit message and an empty result table.

## Geometry and limitations

The second contour level in the supplied pickle format is treated as 90%.
Each contour ring is treated as a filled region. A pair is counted once if the GRB
uncertainty region overlaps at least one ring. Nested rings are not treated as holes.
This is not an exact reconstruction of contour topology.
Each ring must fit within a hemisphere; invalid geometry stops execution rather than
silently reducing sky coverage. RA wrap and repeated closing vertices are handled.
The same geometric predicate is used throughout. Plots use a sky projection only for
display; matching uses spherical angles and minor great-circle segments.

Only trust pickle files from your IceCat-2 data source. Paths must point to your local input files.
Outputs are placed in a timestamped folder. Every observed match gets a two-panel PNG
(sky and time) and is displayed inline. Large numbers of matches can produce long output.


In [ ]:
# ============================================================
# I/O and contours
# ============================================================
def read_final_combined_catalog(path):
    """Read a TSV or CSV file; delimiters are detected automatically."""
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = df.columns.str.strip()
    return df


def read_ic_table(path):
    df = pd.read_csv(path)
    required = {"NAME", "EVENTMJD", "RA [deg]", "DEC [deg]"}
    missing = required - set(df.columns)
    if missing:
        raise RuntimeError(f"Missing columns in the IceCube table: {sorted(missing)}")
    for col in ["EVENTMJD", "RA [deg]", "DEC [deg]"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def ic_ids_from_pkl(pkl_path):
    name = pkl_path.name
    if not (name.startswith("IceCube-") and name.endswith(".contours.pkl")):
        raise ValueError(f"Unexpected PKL filename: {name}")
    ic_id = name.replace("IceCube-", "").replace(".contours.pkl", "")
    return ic_id, "IC" + ic_id


def _collect_rings(obj):
    """Recursively extract all numerical Nx2 rings from one contour level."""
    rings = []
    if isinstance(obj, np.ndarray):
        arr = np.asarray(obj)
        if arr.ndim == 2 and arr.shape[1] >= 2:
            rings.append(np.asarray(arr[:, :2], dtype=float))
        elif arr.dtype == object:
            for item in arr:
                rings.extend(_collect_rings(item))
    elif isinstance(obj, (list, tuple)):
        try:
            arr = np.asarray(obj, dtype=float)
        except (TypeError, ValueError):
            arr = None
        if arr is not None and arr.ndim == 2 and arr.shape[1] >= 2:
            rings.append(arr[:, :2])
        else:
            for item in obj:
                rings.extend(_collect_rings(item))
    return rings


# ============================================================
# Robust spherical geometry
# ============================================================
def radec_to_unit(ra_deg, dec_deg):
    ra = np.deg2rad(np.asarray(ra_deg, dtype=float))
    dec = np.deg2rad(np.asarray(dec_deg, dtype=float))
    cdec = np.cos(dec)
    return np.stack([cdec*np.cos(ra), cdec*np.sin(ra), np.sin(dec)], axis=-1)


def unit_normalize(v, eps=1e-30):
    v = np.asarray(v, dtype=float)
    norm = np.linalg.norm(v, axis=-1, keepdims=True)
    if np.any(norm < eps):
        raise ValueError("A direction vector cannot be normalized.")
    return v / norm


def angdist_u(a_u, b_u):
    dot = np.clip(np.sum(np.asarray(a_u)*np.asarray(b_u), axis=-1), -1.0, 1.0)
    return np.arccos(dot)


def polygon_bounding_cap(poly_u):
    poly = unit_normalize(np.asarray(poly_u, dtype=float))
    if poly.ndim != 2 or poly.shape[1] != 3 or len(poly) < 3:
        raise ValueError("A contour ring must have shape (N, 3), with N >= 3.")
    mean = np.sum(poly, axis=0)
    if np.linalg.norm(mean) < 1e-12:
        raise ValueError("The contour is not localized within one unique hemisphere.")
    center = unit_normalize(mean)
    radius = float(np.max(angdist_u(poly, center)))
    if radius >= np.pi/2:
        raise ValueError("The local contour is too large for a safe projection.")
    return center, radius


def tangent_basis(center_u):
    center = unit_normalize(center_u)
    axes = np.eye(3)
    reference = axes[np.argmin(np.abs(axes @ center))]
    east = unit_normalize(np.cross(reference, center))
    north = np.cross(center, east)
    return east, north


def gnomonic_xy(v_u, projection_center_u):
    """Gnomonic projection; points in the opposite hemisphere are rejected."""
    vectors = unit_normalize(np.asarray(v_u, dtype=float))
    center = unit_normalize(projection_center_u)
    east, north = tangent_basis(center)
    denominator = vectors @ center
    if np.any(denominator <= 1e-12):
        raise ValueError("Point lies outside the projectable hemisphere.")
    x = (vectors @ east) / denominator
    y = (vectors @ north) / denominator
    return np.stack([x, y], axis=-1)


def point_on_planar_segment(point, a, b, tol=1e-12):
    ab = b-a
    ap = point-a
    if np.dot(ab, ab) <= tol**2:
        return np.linalg.norm(ap) <= tol
    scale = max(1.0, np.linalg.norm(ab))
    cross = abs(ab[0]*ap[1] - ab[1]*ap[0])
    return cross <= tol*scale and np.dot(ap, point-b) <= tol*scale


def point_in_planar_polygon(point, polygon):
    """Ray casting including the boundary; independent of polygon orientation."""
    point = np.asarray(point, dtype=float)
    poly = np.asarray(polygon, dtype=float)
    inside = False
    j = len(poly)-1
    for i in range(len(poly)):
        a, b = poly[j], poly[i]
        if point_on_planar_segment(point, a, b):
            return True
        if ((a[1] > point[1]) != (b[1] > point[1])):
            x_cross = (b[0]-a[0])*(point[1]-a[1])/(b[1]-a[1]) + a[0]
            if point[0] < x_cross:
                inside = not inside
        j = i
    return inside


def point_in_spherical_polygon(point_u, poly_u, poly_center_u=None):
    """Safe test for local (< hemisphere) IceCube contours."""
    poly = unit_normalize(np.asarray(poly_u, dtype=float))
    center = polygon_bounding_cap(poly)[0] if poly_center_u is None else poly_center_u
    try:
        poly_xy = gnomonic_xy(poly, center)
        point_xy = gnomonic_xy(np.asarray(point_u)[None, :], center)[0]
    except ValueError:
        return False
    return point_in_planar_polygon(point_xy, poly_xy)


def min_angdist_point_to_gc_segment(c_u, a_u, b_u):
    a, b, c = map(lambda v: unit_normalize(v), (a_u, b_u, c_u))
    normal = np.cross(a, b)
    normal_norm = np.linalg.norm(normal)
    if normal_norm < 1e-15:
        return min(float(angdist_u(c, a)), float(angdist_u(c, b)))
    normal /= normal_norm
    projection = c - np.dot(c, normal)*normal
    if np.linalg.norm(projection) < 1e-15:
        return min(float(angdist_u(c, a)), float(angdist_u(c, b)))
    projection = unit_normalize(projection)
    dab = float(angdist_u(a, b))
    dap = float(angdist_u(a, projection))
    dpb = float(angdist_u(projection, b))
    if abs((dap+dpb)-dab) <= 1e-10:
        return float(angdist_u(c, projection))
    return min(float(angdist_u(c, a)), float(angdist_u(c, b)))


def spherical_cap_polygon_overlap(cap_center_u, cap_radius_rad, ring):
    """Overlap between a GRB error cap and exactly one contour ring."""
    c = unit_normalize(cap_center_u)
    r = float(cap_radius_rad)
    poly = ring["poly_u"]
    poly_center = ring["bound_center_u"]
    if np.any((poly @ c) >= np.cos(r)):
        return True
    if point_in_spherical_polygon(c, poly, poly_center):
        return True
    closed = np.vstack([poly, poly[0]])
    return any(
        min_angdist_point_to_gc_segment(c, a, b) <= r+1e-12
        for a, b in zip(closed[:-1], closed[1:])
    )


def cap_may_overlap_ring(centers_u, radii_rad, ring):
    dots = np.clip(np.asarray(centers_u) @ ring["bound_center_u"], -1.0, 1.0)
    return dots >= np.cos(np.minimum(np.pi, np.asarray(radii_rad) + ring["bound_radius_rad"])) - 1e-14


def cap_overlaps_any_ring(center_u, radius_rad, rings):
    for ring in rings:
        if not cap_may_overlap_ring(center_u[None, :], np.array([radius_rad]), ring)[0]:
            continue
        if spherical_cap_polygon_overlap(center_u, radius_rad, ring):
            return True
    return False


def run_geometry_self_tests():
    square = radec_to_unit([359, 1, 1, 359], [-1, -1, 1, 1])
    center, radius = polygon_bounding_cap(square)
    ring = {"poly_u": square, "bound_center_u": center, "bound_radius_rad": radius}
    assert point_in_spherical_polygon(radec_to_unit(0, 0), square, center)
    assert not point_in_spherical_polygon(radec_to_unit(180, 0), square, center)
    assert cap_overlaps_any_ring(radec_to_unit(0, 0), 0.1*np.pi/180, [ring])
    assert not cap_overlaps_any_ring(radec_to_unit(180, 0), 0.1*np.pi/180, [ring])


# ============================================================
# Time windows
# ============================================================
def make_grb_time_windows(grb_mjd, grb_t90_days, grb_t90_start_days):
    mjd = np.asarray(grb_mjd, dtype=float)
    t90 = np.asarray(grb_t90_days, dtype=float)
    start = np.asarray(grb_t90_start_days, dtype=float)
    if TIME_WINDOW_MODE == "fixed":
        return mjd-FIXED_BEFORE, mjd+FIXED_AFTER
    if TIME_WINDOW_MODE != "t90":
        raise ValueError(f"Unknown TIME_WINDOW_MODE: {TIME_WINDOW_MODE}")

    known_start = np.isfinite(start)
    regular_buffer = np.maximum(T90_BUFFER_FRACTION*t90,
                                MIN_TIME_BUFFER_SECONDS/86400.0)
    replacement_buffer = np.maximum(MISSING_START_BUFFER_FRACTION*t90,
                                    MISSING_START_MIN_SECONDS/86400.0)
    window_start = np.where(known_start,
                            mjd+start-regular_buffer,
                            mjd-replacement_buffer)
    window_end = np.where(known_start,
                          mjd+start+t90+regular_buffer,
                          mjd+t90+replacement_buffer)
    return window_start, window_end


def build_time_mask_for_ic(ic_mjd, grb_start, grb_end):
    return (grb_start <= ic_mjd) & (ic_mjd <= grb_end)


# ============================================================
# Hit counting
# ============================================================
def count_hits(grb_mjd, grb_t90_days, grb_t90_start_days,
               ic_ids, spatial_masks, ic_mjds):
    starts, ends = make_grb_time_windows(grb_mjd, grb_t90_days, grb_t90_start_days)
    return sum(int(np.count_nonzero(
        spatial_masks[ic_id] & build_time_mask_for_ic(ic_mjds[ic_id], starts, ends)
    )) for ic_id in ic_ids)


def count_time_only_hits(grb_mjd, grb_t90_days, grb_t90_start_days, ic_ids, ic_mjds):
    starts, ends = make_grb_time_windows(grb_mjd, grb_t90_days, grb_t90_start_days)
    return sum(int(np.count_nonzero(build_time_mask_for_ic(ic_mjds[i], starts, ends)))
               for i in ic_ids)


def spatial_count_for_random_centers(centers, grb_radii, starts, ends,
                                     ic_ids, ic_mjds, contours):
    total = 0
    for ic_id in ic_ids:
        time_indices = np.where(build_time_mask_for_ic(ic_mjds[ic_id], starts, ends))[0]
        for i in time_indices:
            if cap_overlaps_any_ring(centers[i], grb_radii[i], contours[ic_id]):
                total += 1
    return total


# ============================================================
# Monte Carlo tests
# ============================================================
def permutation_times_test(N, seed, g, ids, masks, mjds):
    rng, counts = np.random.default_rng(seed), np.empty(N, dtype=int)
    indices = np.arange(len(g["mjd"]))
    for k in range(N):
        # Shift each complete GRB time window to another existing trigger.
        shifted_trigger = g["mjd"][rng.permutation(indices)]
        counts[k] = count_hits(shifted_trigger, g["t90_days"], g["t90_start_days"],
                               ids, masks, mjds)
        if (k+1) % 5000 == 0: print(f"Permuted times: {k+1}/{N}")
    return counts


def random_times_test(N, seed, g, ids, masks, mjds):
    rng, counts = np.random.default_rng(seed), np.empty(N, dtype=int)
    tmin, tmax, n = float(np.min(g["mjd"])), float(np.max(g["mjd"])), len(g["mjd"])
    for k in range(N):
        # Only shift the GRB trigger; duration and start offset remain coupled.
        shifted_trigger = rng.uniform(tmin, tmax, n)
        counts[k] = count_hits(shifted_trigger, g["t90_days"], g["t90_start_days"],
                               ids, masks, mjds)
        if (k+1) % 5000 == 0: print(f"Randomized times: {k+1}/{N}")
    return counts


def random_ra_test(N, seed, g, ids, mjds, contours):
    rng, counts = np.random.default_rng(seed), np.empty(N, dtype=int)
    starts, ends = make_grb_time_windows(g["mjd"], g["t90_days"], g["t90_start_days"])
    for k in range(N):
        centers = radec_to_unit(rng.uniform(0, 360, len(g["ra"])), g["dec"])
        counts[k] = spatial_count_for_random_centers(
            centers, g["r90_rad"], starts, ends, ids, mjds, contours)
        if (k+1) % 5000 == 0: print(f"Randomized RA: {k+1}/{N}")
    return counts


def random_ra_and_times_test(N, seed, g, ids, mjds, contours):
    rng, counts = np.random.default_rng(seed), np.empty(N, dtype=int)
    tmin, tmax, n = float(np.min(g["mjd"])), float(np.max(g["mjd"])), len(g["mjd"])
    for k in range(N):
        centers = radec_to_unit(rng.uniform(0, 360, n), g["dec"])
        shifted_trigger = rng.uniform(tmin, tmax, n)
        starts, ends = make_grb_time_windows(
            shifted_trigger, g["t90_days"], g["t90_start_days"])
        counts[k] = spatial_count_for_random_centers(
            centers, g["r90_rad"], starts, ends, ids, mjds, contours)
        if (k+1) % 5000 == 0: print(f"Randomized RA + times: {k+1}/{N}")
    return counts


# ============================================================
# Monte Carlo evaluation
# ============================================================
def summarize_counts(counts, observed):
    counts = np.asarray(counts, dtype=int)
    std = float(np.std(counts, ddof=1)) if len(counts) > 1 else np.nan
    mean = float(np.mean(counts))
    return {"mean": mean, "median": float(np.median(counts)), "std": std,
            "mean ± std": (round(mean-std, 3), round(mean+std, 3)),
            "min": int(np.min(counts)), "max": int(np.max(counts)),
            "p_emp": (int(np.sum(counts >= observed))+1)/(len(counts)+1)}


def print_monte_carlo_summary(counts, observed, label):
    """Print all summary statistics and the complete histogram as plain text."""
    summary = summarize_counts(counts, observed)
    values, frequencies = np.unique(counts, return_counts=True)
    print(f"\n{label}")
    print(f"Monte Carlo runs: {len(counts)}")
    print(f"Observed matches: {observed}")
    print(f"Mean: {summary['mean']}")
    print(f"Median: {summary['median']}")
    print(f"Standard deviation: {summary['std']}")
    print(f"Mean ± standard deviation (rounded bounds): {summary['mean ± std']}")
    print(f"Minimum: {summary['min']}")
    print(f"Maximum: {summary['max']}")
    # Preserve the existing add-one upper-tail estimator.
    print(f"Empirical upper-tail p-value (p_emp): {summary['p_emp']}")
    print("Distribution (coincidences per run: number of runs):")
    for value, frequency in zip(values, frequencies):
        print(f"  {int(value)}: {int(frequency)}")


In [ ]:
"""Input validation, observed-pair reports, and observed-match plots."""

def prepare_grb_data(grb_file, arcsec_to_deg=1.0/3600.0):
    """Keep source rows distinct. Report every exclusion; never invent a duration."""
    df = grb_file.copy() if isinstance(grb_file, pd.DataFrame) else read_final_combined_catalog(grb_file)
    required = {"GRB", "RA", "DEC", "Error Radius 90% [arcsec]", "MJD"}
    if TIME_WINDOW_MODE not in {"t90", "fixed"}:
        raise ValueError("TIME_WINDOW_MODE must be 't90' or 'fixed'.")
    if TIME_WINDOW_MODE == "t90":
        required.add("T90 [s]")
    if required - set(df):
        raise ValueError(f"Missing catalog columns: {sorted(required-set(df))}")
    for col in ["T90 [s]", "T90 Start [s]"]:
        if col not in df:
            df[col] = np.nan
    for col in ["RA", "DEC", "Error Radius 90% [arcsec]", "MJD", "T90 [s]", "T90 Start [s]"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    if "Source row" not in df:
        df["Source row"] = np.arange(len(df))+1
    # A row may have several independent exclusion reasons.
    reasons = {
        "Invalid position": ~(np.isfinite(df.RA) & np.isfinite(df.DEC)
                              & df.RA.between(0, 360) & df.DEC.between(-90, 90)),
        "Invalid 90% radius": ~(np.isfinite(df["Error Radius 90% [arcsec]"])
                                & df["Error Radius 90% [arcsec]"].between(0, 648000, inclusive="right")),
        "Missing/invalid trigger": ~np.isfinite(df.MJD),
    }
    if TIME_WINDOW_MODE == "t90":
        reasons["Missing/invalid T90"] = ~(np.isfinite(df["T90 [s]"]) & (df["T90 [s]"] > 0))
    valid = ~np.logical_or.reduce([v.to_numpy() for v in reasons.values()])
    exclusions = df.loc[~valid].copy()
    exclusions["Exclusion reasons"] = ["; ".join(k for k, v in reasons.items() if v.iloc[i])
                                       for i in np.flatnonzero(~valid)]
    ok = df.loc[valid].reset_index(drop=True)
    print(f"Source rows: {len(df)}; usable: {len(ok)}; excluded: {len(exclusions)}")
    for reason, mask in reasons.items():
        print(f"  {reason}: {int(mask.sum())}")
    g = {"df_all": df, "df_ok": ok, "exclusions": exclusions,
         "names": ok.GRB.astype(str).to_numpy(), "mjd": ok.MJD.to_numpy(float),
         "ra": ok.RA.to_numpy(float), "dec": ok.DEC.to_numpy(float),
         "t90_s": ok["T90 [s]"].to_numpy(float),
         "t90_start_s": ok["T90 Start [s]"].to_numpy(float)}
    g["t90_days"] = g["t90_s"]/86400
    g["t90_start_days"] = g["t90_start_s"]/86400
    g["r90_deg"] = ok["Error Radius 90% [arcsec]"].to_numpy(float)*arcsec_to_deg
    g["r90_rad"] = np.deg2rad(g["r90_deg"])
    g["center_u"] = radec_to_unit(g["ra"], g["dec"])
    return g


def load_90_contours_deg(pkl_path):
    """Read level 1 (90%) of the supplied trusted IceCat-2 pickle format.

    Each ring is treated as filled; overlap with any ring is sufficient.
    Nested rings are not subtracted as holes. This is a conservative
    candidate search, not a reconstruction of contour topology.
    Malformed rings stop the run instead of silently discarding sky coverage.
    """
    with open(pkl_path, "rb") as handle:
        data = pickle.load(handle)
    if not isinstance(data, (tuple, list, np.ndarray)) or len(data) < 2:
        raise ValueError(f"Missing 90% contour level: {pkl_path}")
    raw = _collect_rings(data[1])
    if not raw:
        raise ValueError(f"No 90% rings: {pkl_path}")
    rings = []
    for ring in raw:
        if not np.isfinite(ring).all() or np.any(np.abs(ring[:, 1]) > np.pi/2+1e-10):
            raise ValueError(f"Invalid radian coordinates: {pkl_path}")
        # Remove consecutive duplicate vertices and a repeated closing vertex.
        vectors = radec_to_unit(np.degrees(ring[:, 0]), np.degrees(ring[:, 1]))
        keep = np.r_[True, np.linalg.norm(np.diff(vectors, axis=0), axis=1)>1e-14]
        ring, vectors = ring[keep], vectors[keep]
        if len(ring)>1 and np.linalg.norm(vectors[0]-vectors[-1]) < 1e-14:
            ring = ring[:-1]
        if len(ring) < 3:
            raise ValueError(f"Degenerate 90% ring: {pkl_path}")
        rings.append(np.degrees(ring))
    return rings


def build_spatial_masks(pkl_files, df_ic, grb_data):
    """Use the same spherical overlap test for observations and MC realizations."""
    masks, times, contours = {}, {}, {}
    no_table = 0
    for k, path in enumerate(pkl_files, 1):
        ic_id, name = ic_ids_from_pkl(path)
        rows = df_ic.loc[df_ic.NAME.astype(str).str.strip() == name]
        if rows.empty:
            no_table += 1
            continue
        if len(rows) != 1:
            raise ValueError(f"Ambiguous IceCube table identity: {name}")
        row = rows.iloc[0]
        values = row[["EVENTMJD", "RA [deg]", "DEC [deg]"]].to_numpy(float)
        if not np.isfinite(values).all():
            raise ValueError(f"Invalid IceCube time/position: {name}")
        if ic_id in masks:
            raise ValueError(f"Repeated IceCube contour ID: {ic_id}")
        rings = []
        for xy in load_90_contours_deg(path):
            poly = radec_to_unit(xy[:, 0], xy[:, 1])
            center, radius = polygon_bounding_cap(poly)
            rings.append({"poly_u": poly, "bound_center_u": center,
                          "bound_radius_rad": radius,
                          "ic_center_u": radec_to_unit(values[1], values[2])})
        mask = np.zeros(len(grb_data["mjd"]), dtype=bool)
        for ring in rings:
            idx = np.flatnonzero(~mask & cap_may_overlap_ring(
                grb_data["center_u"], grb_data["r90_rad"], ring))
            for i in idx:
                mask[i] = spherical_cap_polygon_overlap(grb_data["center_u"][i],
                                                       grb_data["r90_rad"][i], ring)
        masks[ic_id], times[ic_id], contours[ic_id] = mask, values[0], rings
        if k % 100 == 0:
            print(f"Processed contours: {k}/{len(pkl_files)}")
    print(f"IceCube events used: {len(masks)}; files absent from IceCube table: {no_table}")
    print(f"90% rings used: {sum(map(len, contours.values()))}; multi-ring events: "
          f"{sum(len(r)>1 for r in contours.values())}")
    if not masks:
        raise RuntimeError("No usable IceCube events. Check input paths and event IDs.")
    return sorted(masks), masks, times, contours


def collect_observed_matches(g, ids, masks, times, label):
    """Count source-row / neutrino pairs once, even when several rings overlap."""
    starts, ends = make_grb_time_windows(g["mjd"], g["t90_days"], g["t90_start_days"])
    records = []
    for ic_id in ids:
        matched = masks[ic_id] & build_time_mask_for_ic(times[ic_id], starts, ends)
        for i in np.flatnonzero(matched):
            row = g["df_ok"].iloc[i]
            records.append({"Catalog": label, "GRB": g["names"][i],
                            "Source row": row["Source row"], "GRB index": int(i),
                            "IceCube ID": ic_id, "Trigger MJD": g["mjd"][i],
                            "Neutrino MJD": times[ic_id],
                            "Delta t [s]": (times[ic_id]-g["mjd"][i])*86400,
                            "Window start MJD": starts[i], "Window end MJD": ends[i],
                            "T90 [s]": g["t90_s"][i], "T90 Start [s]": g["t90_start_s"][i],
                            "RA": g["ra"][i], "DEC": g["dec"][i],
                            "Radius 90% [deg]": g["r90_deg"][i],
                            "Position source": row.get("Position catalog", row.get("best catalog", label))})
    cols = ["Catalog", "GRB", "Source row", "GRB index", "IceCube ID", "Trigger MJD",
            "Neutrino MJD", "Delta t [s]", "Window start MJD", "Window end MJD",
            "T90 [s]", "T90 Start [s]", "RA", "DEC", "Radius 90% [deg]", "Position source"]
    return pd.DataFrame(records, columns=cols)


def sky_projection(vectors, center):
    """Azimuthal equidistant display; matching itself always stays spherical."""
    vectors = unit_normalize(vectors)
    east, north = tangent_basis(center)
    angles = angdist_u(vectors, center)
    sin_angles = np.sin(angles)
    scale = np.divide(angles, sin_angles, out=np.ones_like(angles), where=np.abs(sin_angles)>1e-12)
    return np.rad2deg(np.stack([vectors@east*scale, vectors@north*scale], axis=-1))


def dense_boundary(poly):
    """Interpolate minor great-circle arcs for display, not straight RA/DEC lines."""
    dense = []
    for a,b in zip(poly, np.roll(poly, -1, axis=0)):
        angle = float(angdist_u(a,b))
        n = max(2, int(np.ceil(np.degrees(angle)/0.15)))
        weights = np.linspace(0, 1, n, endpoint=False)
        dense.extend(unit_normalize((1-weights[:,None])*a+weights[:,None]*b))
    return np.vstack([dense, dense[0]])


def plot_observed_matches(matches, g, contours, folder):
    """Plot EVERY observed coincidence (no cap on the number of plots).

    Each figure contains all IC rings, a spherical GRB error-circle boundary,
    both positions, and the temporal window. Figures are saved and closed.
    No Monte Carlo matches are plotted here.
    """
    folder.mkdir(parents=True, exist_ok=True)
    if matches.empty:
        print("No observed spatial-temporal matches to plot.")
        return
    for number, (_, match) in enumerate(matches.iterrows(), 1):
        i = int(match["GRB index"])
        rings = contours[match["IceCube ID"]]
        origin = rings[0]["ic_center_u"]
        fig, (ax, at) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
        for j, ring in enumerate(rings):
            xy = sky_projection(dense_boundary(ring["poly_u"]), origin)
            ax.plot(xy[:,0], xy[:,1], color="royalblue", lw=1,
                    label="IceCube 90% contours" if j==0 else None)
        center, radius = g["center_u"][i], g["r90_rad"][i]
        a,b = tangent_basis(center)
        phi = np.linspace(0, 2*np.pi, 721)
        circle = np.cos(radius)*center + np.sin(radius)*(np.cos(phi)[:,None]*a+np.sin(phi)[:,None]*b)
        xy = sky_projection(circle, origin)
        ax.plot(xy[:,0], xy[:,1], color="darkorange", label="GRB 90% error circle")
        gc = sky_projection(center[None,:], origin)[0]
        ax.scatter(*gc, color="darkorange", marker="x", s=60, label="GRB position")
        ax.scatter(0, 0, color="royalblue", marker="*", s=100, label="IceCube best fit")
        ax.set(xlabel="Local sky coordinate x [deg]", ylabel="Local sky coordinate y [deg]",
               title="Azimuthal equidistant view centered on IceCube")
        ax.set_aspect("equal", adjustable="datalim")
        ax.legend(fontsize=8); ax.grid(alpha=0.25)
        lo = (match["Window start MJD"]-match["Trigger MJD"])*86400
        hi = (match["Window end MJD"]-match["Trigger MJD"])*86400
        at.axvspan(lo, hi, color="gray", alpha=0.2, label="Matching window")
        if TIME_WINDOW_MODE == "t90":
            start = g["t90_start_s"][i]
            # A missing measured start is never labelled as a measured interval.
            if np.isfinite(start):
                at.axvspan(start, start+g["t90_s"][i], alpha=0.35,
                           color="darkorange", label="Measured T90 interval")
            else:
                at.text(0.03, 0.05, "T90 Start unavailable: replacement window", transform=at.transAxes, fontsize=8)
        at.axvline(0, color="black", ls=":", label="GRB trigger")
        at.axvline(match["Delta t [s]"], color="royalblue", lw=2, label="Neutrino time")
        margin = max((hi-lo)*0.1, 1)
        at.set_xlim(min(lo, 0)-margin, max(hi, 0)+margin)
        at.set(xlabel="Time relative to GRB trigger [s]", yticks=[], title="Temporal coincidence")
        at.legend(fontsize=8); at.grid(axis="x", alpha=0.25)
        fig.suptitle(f"{match['Catalog']} | {match['GRB']} (row {match['Source row']}) / IC{match['IceCube ID']}", fontsize=11)
        fig.tight_layout()
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{number:04d}_{match['GRB']}_IC{match['IceCube ID']}")
        fig.savefig(folder / f"{safe}.png", bbox_inches="tight")
        plt.show()
        plt.close(fig)
    print(f"Plotted and saved all {len(matches)} observed pairs in {folder}")


def report_observed(g, ids, masks, times, contours, label, folder):
    folder.mkdir(parents=True, exist_ok=True)
    matches = collect_observed_matches(g, ids, masks, times, label)
    spatial = int(sum(np.count_nonzero(masks[i]) for i in ids))
    temporal = count_time_only_hits(g["mjd"], g["t90_days"], g["t90_start_days"], ids, times)
    print(f"\n{label}: spatial pairs={spatial}; temporal pairs={temporal}; observed pairs={len(matches)}")
    print(f"Distinct matched source rows: {matches['GRB index'].nunique()}; "
          f"matched neutrinos: {matches['IceCube ID'].nunique()}")
    matches.to_csv(folder/"observed_matches.csv", index=False)
    g["exclusions"].to_csv(folder/"excluded_rows.csv", index=False)
    if not matches.empty:
        print(matches.to_string(index=False))
    plot_observed_matches(matches, g, contours, folder/"observed_plots")
    return matches, spatial, temporal


def validate_settings():
    for value in [T90_BUFFER_FRACTION, MIN_TIME_BUFFER_SECONDS,
                  MISSING_START_BUFFER_FRACTION, MISSING_START_MIN_SECONDS, FIXED_BEFORE, FIXED_AFTER]:
        if not np.isfinite(value) or value < 0:
            raise ValueError("Window settings must be finite and non-negative.")
    if not PKL_FILES:
        raise FileNotFoundError(f"No contour pickle files in {PKL_FOLDER}")


def main():
    validate_settings()
    run_geometry_self_tests()
    df_ic = read_ic_table(IC_FILE)
    g = prepare_grb_data(GRB_FILE)
    if not len(g["mjd"]):
        raise ValueError("No usable GRB rows in the combined catalog.")
    ids, masks, times, contours = build_spatial_masks(PKL_FILES, df_ic, g)
    matches, spatial, temporal = report_observed(g, ids, masks, times, contours,
                                               "Combined catalog", OUTPUT_DIR)
    observed = len(matches)
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N must be a positive integer.")
    tests = [
        ("Permuted times", lambda: permutation_times_test(N, SEED, g, ids, masks, times)),
        ("Random times", lambda: random_times_test(N, SEED, g, ids, masks, times)),
        ("Random RA", lambda: random_ra_test(N, SEED, g, ids, times, contours)),
        ("Random RA and times", lambda: random_ra_and_times_test(N, SEED, g, ids, times, contours)),
    ]
    results = {}
    for label, test in tests:
        counts = test()
        results[label] = counts
        pd.DataFrame({"Count": counts}).to_csv(OUTPUT_DIR/(label.replace(" ", "_")+".csv"), index=False)
        print_monte_carlo_summary(counts, observed, label)
    return {"observed_hits": observed, "observed_matches": matches, "spatial_overlaps": spatial,
            "time_only_hits": temporal, "monte_carlo_counts": results, "grb_data": g, "ic_ids": ids}


## Run observations and Monte Carlo

Observed plots are generated before the simulations. The reported mean ± standard deviation is descriptive, not a guaranteed 68% confidence interval. The empirical upper-tail p-value uses an add-one correction.
Monte Carlo results are printed as plain text, including every summary statistic,
the observed count, the run count, and the full frequency distribution. Per-run
counts are still saved as CSV files. No Monte Carlo histogram images are created.


In [ ]:
analysis_results = main()
